# Task 1: Vectorized Scaled Dot-Product Attention from Scratch

## Mathematical Formula
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}} + M\right) V$$


In [1]:
import numpy as np

# Scaled dot-product attention calculation handling 4D tensors and causal masking
def scaled_dot_product_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray, mask: np.ndarray = None):
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.swapaxes(-2, -1)) / np.sqrt(d_k)
    
    if mask is not None:
        scores = np.where(mask == 1, -1e9, scores)
        
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    output = np.matmul(attention_weights, V)
    return output, attention_weights


In [2]:
# Initialize mini dataset of sequence tokens and run multi-head attention
tokens = ["The", "quick", "brown", "fox", "jumps", "over"]
batch_size, num_heads, seq_len, head_dim = 2, 4, len(tokens), 16

np.random.seed(42)
Q = np.random.randn(batch_size, num_heads, seq_len, head_dim)
K = np.random.randn(batch_size, num_heads, seq_len, head_dim)
V = np.random.randn(batch_size, num_heads, seq_len, head_dim)

causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1)
output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

print("Input Sequence Tokens:", tokens)
print("Query Matrix Shape:", Q.shape)
print("Key Matrix Shape:", K.shape)
print("Value Matrix Shape:", V.shape)
print("Weighted Context Output Shape:", output.shape)
print("\nAttention Probabilities for Head 0 (Token 'jumps' attending to preceding tokens):")
for i, tok in enumerate(tokens):
    print(f"  Token '{tok}': {attn_weights[0, 0, 4, i]:.4f}")


Input Sequence Tokens: ['The', 'quick', 'brown', 'fox', 'jumps', 'over']
Query Matrix Shape: (2, 4, 6, 16)
Key Matrix Shape: (2, 4, 6, 16)
Value Matrix Shape: (2, 4, 6, 16)
Weighted Context Output Shape: (2, 4, 6, 16)

Attention Probabilities for Head 0 (Token 'jumps' attending to preceding tokens):
  Token 'The': 0.0854
  Token 'quick': 0.4848
  Token 'brown': 0.1577
  Token 'fox': 0.1725
  Token 'jumps': 0.0996
  Token 'over': 0.0000
